In [1]:
import pandas as pd
import numpy as np

In [2]:
# Chapter 10. Data Aggregation and Group Operations

In [3]:
# pandas provides a versatile .groupby() interface, enabling to slice and summarize datasets in a natural way
# query languages impose certain limitations on the kinds of group operations
# with the Python expressiveness can express as custom Python functions 

In [4]:
# 10.1 How to think About Group Operations

In [5]:
# group operations : split-apply-combine
# Series, DataFrame is split into groups based on one or more keys that you provide
# splitting is performed on a particular axis of an object
# once split, function is applied to each group, producing a new value
# results of all those function applications are combined into a result object

In [6]:
# each grouping key can take many forms, and the keys do not have to be all the same type:
# - A list or array of values that is the same length as the axis being grouped
# - A value indicating a column name in a DataFrame
# - A dictionary or Series giving a correspondence between the values on the axis being grouped and the group names
# - A function to be invoked on the axis index or the individuals labels in the index

In [7]:
df = pd.DataFrame({"key1":["a","a",None,"b","b","a",None],
                  "key2":pd.Series([1,2,1,2,1,None,1],dtype="Int64"),
                  "data1":np.random.standard_normal(7),
                  "data2":np.random.standard_normal(7)})
df

,key1,key2,data1,data2
0,a,1,1.490291,0.354791
1,a,2,0.125183,1.130729
2,None,1,0.835775,0.585008
3,b,2,-0.597081,-0.426038
4,b,1,0.683163,1.237802
5,a,<NA>,-1.047222,1.146763
6,None,1,0.847094,0.689127


In [8]:
# suppose you want to compute the mean of the data1 column using the labels from key1

In [9]:
# method 1 : access data1 and call groupby with the column (a Series) at key1:

grouped = df["data1"].groupby(df["key1"])
grouped

In [10]:
# grouped is now a special "GroupBy" object
# to compute group means we can call the GroupBy's .mean() method:
grouped.mean()

# here the data (a Series) has been aggregated by splitting the data on the group key;
# producing a new Series that is now indexed by the unique values in the key1 column
# the result index has the name "key1" because the DataFrame column df["key1"] did

key1
a    0.189417
b    0.043041
Name: data1, dtype: float64

In [11]:
# if instead passed multiple arrays as a list, we'd get something different:
means = df["data1"].groupby([df["key1"],df["key2"]]).mean()
means

key1  key2
a     1       1.490291
      2       0.125183
b     1       0.683163
      2      -0.597081
Name: data1, dtype: float64

In [12]:
# here we grouped the data using two keys, and the resulting Series now has a heirarchical index;
# which consist of the unique pairs of keys observed:
means.unstack()

key2,1,2
key1,,
a,1.490291,0.125183
b,0.683163,-0.597081


In [13]:
# in this example, the group keys are all Series, though they could be any arrays of the right lenght:
states = np.array(["OH","CA","CA","OH","OH","CA","OH"])
years = [2005,2005,2006,2005,2006,2005,2006]
df["data1"].groupby([states,years]).mean()

CA  2005   -0.461020
    2006    0.835775
OH  2005    0.446605
    2006    0.765128
Name: data1, dtype: float64

In [14]:
# frequently, the grouping information is the same DataFrame as the data you want to work on
# in this caes, you can pass column names as the group keys:
df.groupby("key1").mean()

,key2,data1,data2
key1,,,
a,1.5,0.189417,0.877428
b,1.5,0.043041,0.405882


In [15]:
df.groupby("key2").mean()

# there is no key1 column in the result since df["key1"] is not numeric data
# non-numeric data is said to be a nuisance column which is automatically excluded from result
# by default, all of the numeric columns are aggregated

,data1,data2
key2,,
1,0.964081,0.716682
2,-0.235949,0.352346


In [16]:
df.groupby(["key1","key2"]).mean()

data1     data2
key1 key2                    
a    1     1.490291  0.354791
     2     0.125183  1.130729
b    1     0.683163  1.237802
     2    -0.597081 -0.426038

In [17]:
# regardless of the objective in using .groupby(), generally useful GroupBy method is .size()
# .size() returns a Series containing group sizes:

df.groupby(["key1","key2"]).size()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

In [18]:
# any missing values in a group key are excluded from the result by default
# this behavior can be disabled by passing dropna=False to .groupby:

df.groupby("key1",dropna=False).size()

key1
a      3
b      2
NaN    2
dtype: int64

In [19]:
df.groupby(["key1","key2"],dropna=False).size()

key1  key2
a     1       1
      2       1
      NaN     1
b     1       1
      2       1
NaN   1       2
dtype: int64

In [20]:
# a group function similar to .size() is .count(), which computes the number of non-null values in each group:

df.groupby("key1").count()

,key2,data1,data2
key1,,,
a,2,3,3
b,2,2,2


In [21]:
# Iterating over Groups
# the object returned by .groupby() supports iteration, generating a sequence of 2-tuples containing the group name along with the chunk of data

df

,key1,key2,data1,data2
0,a,1,1.490291,0.354791
1,a,2,0.125183,1.130729
2,None,1,0.835775,0.585008
3,b,2,-0.597081,-0.426038
4,b,1,0.683163,1.237802
5,a,<NA>,-1.047222,1.146763
6,None,1,0.847094,0.689127


In [22]:
for fuck,shit in df.groupby("key1"):
    print(fuck) # print the label, which is indexes under key1
    print(shit) # print the corresponding data to the label

# for loop above uses 2 variables - 1 for label and 1 for the corresponding data
# .groupby() requires 2 variables in for loop, since .groupby yields a tuple containing (Label, Data)

a
  key1  key2     data1     data2
0    a     1  1.490291  0.354791
1    a     2  0.125183  1.130729
5    a  <NA> -1.047222  1.146763
b
  key1  key2     data1     data2
3    b     2 -0.597081 -0.426038
4    b     1  0.683163  1.237802


In [23]:
# in the case of multiple keys, the first elemnt in the tuple will be a tuple of key values:
for(k1,k2),group in df.groupby(["key1","key2"]):
    print((k1,k2))
    print(group)

# for loop above : (k1,k2) and group are variables

('a', 1)
  key1  key2     data1     data2
0    a     1  1.490291  0.354791
('a', 2)
  key1  key2     data1     data2
1    a     2  0.125183  1.130729
('b', 1)
  key1  key2     data1     data2
4    b     1  0.683163  1.237802
('b', 2)
  key1  key2     data1     data2
3    b     2 -0.597081 -0.426038


In [24]:
# you can choose to do whatever you want with the pieces of data
# computing a dictionary of the data pieces as a one-liner:
pieces = {name:group for name,group in df.groupby("key1")}
pieces["b"]

,key1,key2,data1,data2
3,b,2,-0.597081,-0.426038
4,b,1,0.683163,1.237802


In [25]:
# by default .groupby() groups on axis="index" but you can group on any other of the axes
# grouping the columns of the df above by whether they start with "key" or "data":

grouped = df.groupby({"key1":"key","key2":"key","data1":"data","data2":"data"},axis="columns")
grouped

In [26]:
# we can print out the above groups like so:

for group_key,group_values in grouped:
    print(group_key)
    print(group_values)

data
      data1     data2
0  1.490291  0.354791
1  0.125183  1.130729
2  0.835775  0.585008
3 -0.597081 -0.426038
4  0.683163  1.237802
5 -1.047222  1.146763
6  0.847094  0.689127
key
   key1  key2
0     a     1
1     a     2
2  None     1
3     b     2
4     b     1
5     a  <NA>
6  None     1


In [27]:
# Grouping with Dictionaries and Series

# grouping information may exist in a form other than an array

people = pd.DataFrame(np.random.standard_normal((5,5)),
                      columns=["a","b","c","d","e"],
                      index=["Joe","Steve","Wanda","Jill","Trey"])
people

,a,b,c,d,e
Joe,-1.109985,0.538739,0.118286,1.528091,-0.634333
Steve,-1.181710,0.186526,-0.124582,-0.712097,-0.704521
Wanda,2.357812,-0.540936,0.525405,0.433397,-1.028363
Jill,-1.907636,-1.329051,-0.445600,-2.268683,1.684697
Trey,-0.764562,-0.068417,-0.814363,-1.213954,0.825992


In [28]:
people.iloc[2:3,[1,2]]=np.nan # adding a few NaN values
people

,a,b,c,d,e
Joe,-1.109985,0.538739,0.118286,1.528091,-0.634333
Steve,-1.181710,0.186526,-0.124582,-0.712097,-0.704521
Wanda,2.357812,NaN,NaN,0.433397,-1.028363
Jill,-1.907636,-1.329051,-0.445600,-2.268683,1.684697
Trey,-0.764562,-0.068417,-0.814363,-1.213954,0.825992


In [29]:
# suppose we get a group correspondence for the coluns and want to sum the columns by group:

mapping = {"a":"red","b":"red","c":"blue","d":"blue","e":"red","f":"orange"}
mapping

{'a': 'red', 'b': 'red', 'c': 'blue', 'd': 'blue', 'e': 'red', 'f': 'orange'}

In [30]:
# now, construct an array from this dictionary to pass to a .groupby()
# but instead we can just pass the dictionary (additional key "f" has been added to show unused grouping keys are okay)

by_column = people.groupby(mapping,axis="columns")
by_column

In [31]:
by_column.sum()

# this is like conditional sum. By assiging grouping logic, given in dictionary, data can be grouped and aggregated.

,blue,red
Joe,1.646377,-1.205578
Steve,-0.836679,-1.699704
Wanda,0.433397,1.329449
Jill,-2.714284,-1.551990
Trey,-2.028317,-0.006987


In [32]:
# the same functionality holds for Series, which can be viewed as a fixed-size mapping:

map_series = pd.Series(mapping)
map_series

a       red
b       red
c      blue
d      blue
e       red
f    orange
dtype: object

In [33]:
people.groupby(map_series,axis="columns").count()

,blue,red
Joe,2,3
Steve,2,3
Wanda,1,2
Jill,2,3
Trey,2,3


In [34]:
# Grouping with Functions

# defining functions for group mapping
# any function passed as a group key will be called once per index value with the return values being used as the group names
# suppose you want to group by name length - in this case passing a .len() works:

people.groupby(len).sum()

,a,b,c,d,e
3,-1.109985,0.538739,0.118286,1.528091,-0.634333
4,-2.672198,-1.397469,-1.259963,-3.482638,2.510689
5,1.176102,0.186526,-0.124582,-0.278700,-1.732883


In [35]:
# mixing functions with arrays, dictionaries, or Series is not a problem, as everything can be converted to arrays internally:

key_list = ["one","one","one","two","two"]
key_list

['one', 'one', 'one', 'two', 'two']

In [36]:
people

,a,b,c,d,e
Joe,-1.109985,0.538739,0.118286,1.528091,-0.634333
Steve,-1.181710,0.186526,-0.124582,-0.712097,-0.704521
Wanda,2.357812,NaN,NaN,0.433397,-1.028363
Jill,-1.907636,-1.329051,-0.445600,-2.268683,1.684697
Trey,-0.764562,-0.068417,-0.814363,-1.213954,0.825992


In [37]:
people.groupby([len,key_list]).min()

# grouping together: lenght of indexes & given key_list
# grouping results become tuples:
# [Joe,3,one] --> (3,"one")
# [Steve,5,one] --> (5,"one")
# [Wanda,5,one] --> (5,"one")
# [Jill,4,two] --> (4,"two")
# [Trey,4,two] --> (4,"two")

,,a,b,c,d,e
3,one,-1.109985,0.538739,0.118286,1.528091,-0.634333
4,two,-1.907636,-1.329051,-0.814363,-2.268683,0.825992
5,one,-1.181710,0.186526,-0.124582,-0.712097,-1.028363


In [38]:
# Grouping by Index Levels

# aggregating using one of the levels of an axis index

columns = pd.MultiIndex.from_arrays([["US","US","US","JP","JP"],
                                     [1,3,5,1,3]],
                                    names=["cty","tenor"])
columns

MultiIndex([('US', 1),
            ('US', 3),
            ('US', 5),
            ('JP', 1),
            ('JP', 3)],
           names=['cty', 'tenor'])

In [39]:
hier_df = pd.DataFrame(np.random.standard_normal((4,5)),columns=columns) # note that calling columns it's the object columns, not "columns"
hier_df

cty          US                            JP          
tenor         1         3         5         1         3
0      0.133770  1.318378 -1.765161 -0.677366  2.959750
1     -0.478229  0.492227  0.273312  0.225176 -0.828910
2     -0.035755  0.309817 -0.326200  0.152188 -1.288710
3      0.899723 -0.341526 -0.503760  0.605252  1.516387

In [40]:
# to group by level, pass the level number or name using the "level" keyword:

hier_df.groupby(level="cty",axis="columns").count()

cty,JP,US
0,2,3
1,2,3
2,2,3
3,2,3


In [41]:
# 10.2 Data Aggregation

# Aggregations refer to any data transformation that produces scalar values from arrays - mean, count, min, and sum

# Optimized common aggregations groupby methods
# .any(), .all() : Return True if any (one or more values) or all non-NA values are "truthy"
# .count() : Number of non-NA values
# .cummin(), .cummax() : Cumulative minimum and maximum of non-NA values
# .cumsum() : Cumulative sum of non-NA values
# .cumprod() : Cumulative product of non-NA values
# .first(), .last() : First and Last non-NA values
# .mean() : Mean of non-NA values
# .median() : Arithmetic median of non-NA values
# .min(), .max() : Minimum and Maximum of non-NA values
# .nth() : Retrieve value that would appear at position n with the data in sorted order
# .ohlc() : Compute four "open-high-low-close" statistics for time series-like data
# .prod() : Product of non-NA values
# .quantile() : Compute sample quantile
# .rank() : Ordinal ranks of non-NA values, like calling Series.rank()
# .size() : Compute group size, returning result as a Series
# .std(), .var() : Sample standard deviation and variance

In [42]:
# we can also use aggregrations of our own, even though not explicitly implemented for Groupby
# we can use .nsmallets()
# internally, Groupby slices up the Series, calls piece.nsmallest(n) for each piece, and then assembles those results into the result object:

df

,key1,key2,data1,data2
0,a,1,1.490291,0.354791
1,a,2,0.125183,1.130729
2,None,1,0.835775,0.585008
3,b,2,-0.597081,-0.426038
4,b,1,0.683163,1.237802
5,a,<NA>,-1.047222,1.146763
6,None,1,0.847094,0.689127


In [43]:
grouped = df.groupby("key1")
grouped

In [44]:
grouped["data1"].nsmallest(2) # .nsmallest() returns the first n rows with the smallest values in a specified column

# it can be like .nsmallest() == .sort_values() (Ascending) and head(n) combined together

key1   
a     5   -1.047222
      1    0.125183
b     3   -0.597081
      4    0.683163
Name: data1, dtype: float64

In [45]:
# to use your own aggregation functions, pass any function that aggregates an array to the aggregate method or its short alias agg:

def peak_to_peak(arr):
    return arr.max() - arr.min()

In [46]:
grouped.agg(peak_to_peak)

,key2,data1,data2
key1,,,
a,1,2.537514,0.791973
b,1,1.280243,1.663840


In [47]:
# you may notice that some methods, like .describe() also work, even though they are not aggregations:

grouped.describe()

key2                                           data1            ...  \
     count mean       std  min   25%  50%   75%  max count      mean  ...   
key1                                                                  ...   
a      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   3.0  0.189417  ...   
b      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   2.0  0.043041  ...   

                         data2                                          \
           75%       max count      mean       std       min       25%   
key1                                                                     
a     0.807737  1.490291   3.0  0.877428  0.452688  0.354791  0.742760   
b     0.363102  0.683163   2.0  0.405882  1.176513 -0.426038 -0.010078   

                                    
           50%       75%       max  
key1                                
a     1.130729  1.138746  1.146763  
b     0.405882  0.821842  1.237802  

[2 rows x 24 columns]

In [48]:
# Note : Custom aggregation functions are generally much slower than the optimized functions 
#        this is because there is some extra overhead (function cells, data rearrangement) in constructing the intermediate group data chunks

In [50]:
# Column-Wise and Multiple Function Application

tips = pd.read_csv("examples/tips.csv")
tips.head()

,total_bill,tip,smoker,day,time,size
0,16.99,1.01,No,Sun,Dinner,2
1,10.34,1.66,No,Sun,Dinner,3
2,21.01,3.50,No,Sun,Dinner,3
3,23.68,3.31,No,Sun,Dinner,2
4,24.59,3.61,No,Sun,Dinner,4


In [51]:
# add a tip_pct column with the tipe percentage of the total bill:

tips["tip_pct"] = tips["tip"]/tips["total_bill"]
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [52]:
# aggregating a Series or all of the columns of a DataFrame is a matter of using agg with desired function
# however, you may want to agg using a different function, depending on the column, or multiple functions at once
# first, group the tips by day and smoker:

grouped = tips.groupby(["day","smoker"])

In [53]:
# Note : for descriptive statistics, you can pass the name of the function as a string, within .agg()

grouped_pct = grouped["tip_pct"]

In [54]:
grouped_pct.agg("mean")

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

In [55]:
# if you pass a list of functions or function names instead, you get back a DataFrame with column names taken from the functions:

grouped_pct.agg(["mean","std",peak_to_peak])

mean       std  peak_to_peak
day  smoker                                  
Fri  No      0.151650  0.028123      0.067349
     Yes     0.174783  0.051293      0.159925
Sat  No      0.158048  0.039767      0.235193
     Yes     0.147906  0.061375      0.290095
Sun  No      0.160113  0.042347      0.193226
     Yes     0.187250  0.154134      0.644685
Thur No      0.160298  0.038774      0.193350
     Yes     0.163863  0.039389      0.151240

In [56]:
# you don't need to accept the names that GroupBy gives to the columns; notably, lambda functions have the name "<lambda>", which is hard to identify
# if you pass a list of (name,function) tuples, the first element of each tuple will be used as the DataFrame column names
# in tuple format .agg([("name","function"),("name","function")])

grouped_pct.agg([("average","mean"),("stdev",np.std)])

average     stdev
day  smoker                    
Fri  No      0.151650  0.028123
     Yes     0.174783  0.051293
Sat  No      0.158048  0.039767
     Yes     0.147906  0.061375
Sun  No      0.160113  0.042347
     Yes     0.187250  0.154134
Thur No      0.160298  0.038774
     Yes     0.163863  0.039389

In [57]:
# with a DataFrame you have more options, as you can specify a list of functions to apply to all of the columns or different functions per column
# suppose we wanted to compute the same three statistics for the tip_pct and total_bill columns:

functions = ["count","mean","max"]

In [58]:
result = grouped[["tip_pct","total_bill"]].agg(functions)
result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

In [59]:
# the result DataFrame has hierarchical columns, the same you would get aggregating each column separately and using concat to glue the results together using the column names as the keys argument:
result["tip_pct"]

count      mean       max
day  smoker                           
Fri  No          4  0.151650  0.187735
     Yes        15  0.174783  0.263480
Sat  No         45  0.158048  0.291990
     Yes        42  0.147906  0.325733
Sun  No         57  0.160113  0.252672
     Yes        19  0.187250  0.710345
Thur No         45  0.160298  0.266312
     Yes        17  0.163863  0.241255

In [60]:
# a list of tuples with custom names can be passed:

ftuples = [("average","mean"),("Variance",np.var)]

In [61]:
grouped[["tip_pct","total_bill"]].agg(ftuples)

tip_pct           total_bill            
              average  Variance    average    Variance
day  smoker                                           
Fri  No      0.151650  0.000791  18.420000   25.596333
     Yes     0.174783  0.002631  16.813333   82.562438
Sat  No      0.158048  0.001581  19.661778   79.908965
     Yes     0.147906  0.003767  21.276667  101.387535
Sun  No      0.160113  0.001793  20.506667   66.099980
     Yes     0.187250  0.023757  24.120000  109.046044
Thur No      0.160298  0.001503  17.113111   59.625081
     Yes     0.163863  0.001551  19.190588   69.808518

In [62]:
# suppose you wanted to apply potentially diferent funcitons to one or more of the columns
# pass a dictionary to .agg() that contains a mapping of column names to any of the function specifications listed so far:

grouped.agg({"tip":np.max,"size":"sum"})

tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [63]:
grouped.agg({"tip_pct":["min","max","mean","std"],
             "size":"sum"})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.120385  0.187735  0.151650  0.028123    9
     Yes     0.103555  0.263480  0.174783  0.051293   31
Sat  No      0.056797  0.291990  0.158048  0.039767  115
     Yes     0.035638  0.325733  0.147906  0.061375  104
Sun  No      0.059447  0.252672  0.160113  0.042347  167
     Yes     0.065660  0.710345  0.187250  0.154134   49
Thur No      0.072961  0.266312  0.160298  0.038774  112
     Yes     0.090014  0.241255  0.163863  0.039389   40

In [64]:
# a DataFrame will have hierarchical columns only if multiple funcitons are applied to at least one column

In [65]:
# Returning Aggregated Data Without Row Indexes

# you can disable the aggregated data coming with an index, in most cases, by passing "as_index=False" to .groupby():

tips

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
...,...,...,...,...,...,...,...
239,29.03,5.92,No,Sat,Dinner,3,0.203927
240,27.18,2.00,Yes,Sat,Dinner,2,0.073584
241,22.67,2.00,Yes,Sat,Dinner,2,0.088222
242,17.82,1.75,No,Sat,Dinner,2,0.098204


In [66]:
tips.groupby(["day","smoker"],as_index=False).mean(numeric_only=True)

# ** Pandas has updated and become more rigid. In older version, it would automatically drop text (string) columns
#     but with update, the automatic text drop process is gone, and need to explicitly specify "numeric_only=True"

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


In [67]:
# 10.3 Apply: General split-apply-combine

# the most general-purpose GroupBy method is .apply()
# .apply() splits the object being manipulated into pieces, invokes the passed function on each piece, and then attempts to concatenate the pieces


In [68]:
# tips dataset, suppose you wanted to select the top five tip_pct values by group
# first, write a function that selects the rows with the largest values in a particular column:

def top(df,n=5,column="tip_pct"):
    return df.sort_values(column,ascending=False)[:n]

In [69]:
top(tips,n=6)

,total_bill,tip,smoker,day,time,size,tip_pct
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
232,11.61,3.39,No,Sat,Dinner,2,0.291990
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


In [70]:
# now, if we group by smoker, and cally apply with this function, we get the following:

tips.groupby("smoker").apply(top)

total_bill   tip smoker   day    time  size   tip_pct
smoker                                                           
No     232       11.61  3.39     No   Sat  Dinner     2  0.291990
       149        7.51  2.00     No  Thur   Lunch     2  0.266312
       51        10.29  2.60     No   Sun  Dinner     2  0.252672
       185       20.69  5.00     No   Sun  Dinner     5  0.241663
       88        24.71  5.85     No  Thur   Lunch     2  0.236746
Yes    172        7.25  5.15    Yes   Sun  Dinner     2  0.710345
       178        9.60  4.00    Yes   Sun  Dinner     2  0.416667
       67         3.07  1.00    Yes   Sat  Dinner     1  0.325733
       183       23.17  6.50    Yes   Sun  Dinner     4  0.280535
       109       14.31  4.00    Yes   Sat  Dinner     2  0.279525

In [71]:
# to explain, first, the tips DataFrame is split into groups based on the value of smoker column.
# then the top function is called on each group, and the results of each function call are glued together using pd.concat(),
# labeling the pieces with the group names.
# the result therefore has a heirarchical index with an inner level that contains index values from the original DF

# if you pass a function to apply that takes other arguments or keywords, you can pass these after the function:

tips.groupby(["smoker","day"]).apply(top,n=1,column="total_bill")

total_bill    tip smoker   day    time  size   tip_pct
smoker day                                                             
No     Fri  94        22.75   3.25     No   Fri  Dinner     2  0.142857
       Sat  212       48.33   9.00     No   Sat  Dinner     4  0.186220
       Sun  156       48.17   5.00     No   Sun  Dinner     6  0.103799
       Thur 142       41.19   5.00     No  Thur   Lunch     5  0.121389
Yes    Fri  95        40.17   4.73    Yes   Fri  Dinner     4  0.117750
       Sat  170       50.81  10.00    Yes   Sat  Dinner     3  0.196812
       Sun  182       45.35   3.50    Yes   Sun  Dinner     3  0.077178
       Thur 197       43.11   5.00    Yes  Thur   Lunch     4  0.115982

In [74]:
# what occurs inside the function passed is up to you; it must either return a pandas object or a scalar value
# here are various examples showing how to solve various problems using groupby

result = tips.groupby("smoker")["tip_pct"].describe()
result

,count,mean,std,min,25%,50%,75%,max
smoker,,,,,,,,
No,151.0,0.159328,0.039910,0.056797,0.136906,0.155625,0.185014,0.291990
Yes,93.0,0.163196,0.085119,0.035638,0.106771,0.153846,0.195059,0.710345


In [75]:
result.unstack("smoker")

       smoker
count  No        151.000000
       Yes        93.000000
mean   No          0.159328
       Yes         0.163196
std    No          0.039910
       Yes         0.085119
min    No          0.056797
       Yes         0.035638
25%    No          0.136906
       Yes         0.106771
50%    No          0.155625
       Yes         0.153846
75%    No          0.185014
       Yes         0.195059
max    No          0.291990
       Yes         0.710345
dtype: float64

In [76]:
# inside GroupBy, when you invoke a method like .describe(), it is actually just a shortcut for:

def f(group):
    return group.describe()

grouped.apply(f)

total_bill       tip  size   tip_pct
day  smoker                                            
Fri  No     count    4.000000  4.000000  4.00  4.000000
            mean    18.420000  2.812500  2.25  0.151650
            std      5.059282  0.898494  0.50  0.028123
            min     12.460000  1.500000  2.00  0.120385
            25%     15.100000  2.625000  2.00  0.137239
...                       ...       ...   ...       ...
Thur Yes    min     10.340000  2.000000  2.00  0.090014
            25%     13.510000  2.000000  2.00  0.148038
            50%     16.470000  2.560000  2.00  0.153846
            75%     19.810000  4.000000  2.00  0.194837
            max     43.110000  5.000000  4.00  0.241255

[64 rows x 4 columns]

In [77]:
# Suppressing the Group Keys

# earlier, you see that the resulting object has a hierarchical index formed from the group keys, along with the indexes of each piece of the original object
# you can disable this by passing group_keys=False to .groupby():

tips.groupby("smoker",group_keys=False).apply(top)

,total_bill,tip,smoker,day,time,size,tip_pct
232,11.61,3.39,No,Sat,Dinner,2,0.291990
149,7.51,2.00,No,Thur,Lunch,2,0.266312
51,10.29,2.60,No,Sun,Dinner,2,0.252672
185,20.69,5.00,No,Sun,Dinner,5,0.241663
88,24.71,5.85,No,Thur,Lunch,2,0.236746
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


In [79]:
# Quantile and Bucket Analysis

# from chapter 8, pd.cut() & pd.qcut() for slicing data up into buckets with bins of your choosing
# consider a simple random dataset and an equal-length bucket categorization using pd.cut():

frame = pd.DataFrame({"data1":np.random.standard_normal(1000),
                      "data2":np.random.standard_normal(1000)})
frame.head()

,data1,data2
0,1.056279,0.185877
1,-0.297542,-0.815152
2,-0.730554,0.040723
3,-0.027169,-0.321704
4,-1.183414,0.713004


In [80]:
quartiles = pd.cut(frame["data1"],4)
quartiles.head(10)

0       (0.102, 1.7]
1    (-1.496, 0.102]
2    (-1.496, 0.102]
3    (-1.496, 0.102]
4    (-1.496, 0.102]
5       (0.102, 1.7]
6    (-1.496, 0.102]
7    (-1.496, 0.102]
8     (-3.1, -1.496]
9    (-1.496, 0.102]
Name: data1, dtype: category
Categories (4, interval[float64, right]): [(-3.1, -1.496] < (-1.496, 0.102] < (0.102, 1.7] < (1.7, 3.298]]

In [81]:
# the Categorical object returned by cut can be passed directly to .groupby()
# so we could compute a set of group statistics for the qurartiles, like:

def get_stats(group):
    return pd.DataFrame(
        {"min":group.min(),"max":group.max(),
         "count":group.count(),"mean":group.mean()}
    )

In [82]:
grouped = frame.groupby(quartiles)
grouped

In [84]:
grouped.apply(get_stats)

min       max  count      mean
data1                                                     
(-3.1, -1.496]  data1 -3.093647 -1.497860     73 -1.950294
                data2 -2.588668  3.282430     73  0.126584
(-1.496, 0.102] data1 -1.484846  0.101482    471 -0.555250
                data2 -2.966916  2.963348    471  0.021929
(0.102, 1.7]    data1  0.104791  1.699934    411  0.755441
                data2 -2.889460  2.775765    411 -0.071217
(1.7, 3.298]    data1  1.703512  3.298145     45  2.129935
                data2 -2.656905  1.097451     45 -0.207351

In [85]:
# keep in mind the same result could have been computed more simply with:

grouped.agg(["min","max","count","mean"])

data1                               data2                  \
                      min       max count      mean       min       max count   
data1                                                                           
(-3.1, -1.496]  -3.093647 -1.497860    73 -1.950294 -2.588668  3.282430    73   
(-1.496, 0.102] -1.484846  0.101482   471 -0.555250 -2.966916  2.963348   471   
(0.102, 1.7]     0.104791  1.699934   411  0.755441 -2.889460  2.775765   411   
(1.7, 3.298]     1.703512  3.298145    45  2.129935 -2.656905  1.097451    45   

                           
                     mean  
data1                      
(-3.1, -1.496]   0.126584  
(-1.496, 0.102]  0.021929  
(0.102, 1.7]    -0.071217  
(1.7, 3.298]    -0.207351

In [86]:
# to compute equal-size buckets based on sample quantiles, use pd.qcut()
# we can pass 4 as the number of bucket compute sample quartiles, and pass labels=False to obtain just the quartile indices instead of intervals:

quartiles_samp = pd.qcut(frame["data1"],4,labels=False)
quartiles_samp.head()

0    3
1    1
2    0
3    2
4    0
Name: data1, dtype: int64